# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [50]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [51]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [52]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [53]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [54]:
from pyspark.sql import functions as F

df_trips.printSchema()
df_trips.show(5)
print("rows:", df_trips.count())

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+---

In [55]:
#Add a column that creates a unique key to identify each record in order to answer questions about individual trips
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())

df_trips = df_trips.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime")
     - F.unix_timestamp("tpep_pickup_datetime")) / 60
)
df_trips.select("trip_id", "tpep_pickup_datetime", "duration_min").show(5)

+-----------+--------------------+------------------+
|    trip_id|tpep_pickup_datetime|      duration_min|
+-----------+--------------------+------------------+
|60129542144| 2019-01-01 00:46:40| 6.666666666666667|
|60129542145| 2019-01-01 00:59:47|              19.2|
|60129542146| 2018-12-21 13:48:30| 4.166666666666667|
|60129542147| 2018-11-28 15:52:25|3.3333333333333335|
|60129542148| 2018-11-28 15:56:57|               1.6|
+-----------+--------------------+------------------+
only showing top 5 rows


In [56]:
#Which trip has the highest passanger count
max_pc = df_trips.agg(F.max("passenger_count")).first()[0]
print("max passengers:", max_pc)

(df_trips.filter(F.col("passenger_count")== max_pc).select("trip_id", "passenger_count", "trip_distance", "total_amount").show())

max passengers: 9.0
+-----------+---------------+-------------+------------+
|    trip_id|passenger_count|trip_distance|total_amount|
+-----------+---------------+-------------+------------+
|60130492100|            9.0|          0.0|        12.6|
|60130838431|            9.0|          0.0|         9.3|
|60131554242|            9.0|          0.0|        11.3|
|60132426139|            9.0|          0.0|       12.25|
|60134076851|            9.0|          0.0|      110.76|
|60134394369|            9.0|          0.0|       12.74|
|60134539934|            9.0|          0.0|         9.8|
|60136828827|            9.0|          0.0|        10.3|
|60136916020|            9.0|        13.38|        90.8|
+-----------+---------------+-------------+------------+



In [57]:
#What is the Average passanger count
df_trips.agg(
    F.round(F.avg("passenger_count"), 2).alias("avg_passengers")
).show()

+--------------+
|avg_passengers|
+--------------+
|          1.57|
+--------------+



In [58]:
cols = ["trip_id", "trip_distance", "duration_min",
        "tpep_pickup_datetime", "tpep_dropoff_datetime"]

print("Shortest by distance")
df_trips.orderBy("trip_distance").select(cols).show(5)
print("Shortest real trip (distance > 0)")
(df_trips.filter(F.col("trip_distance") > 0)
    .orderBy("trip_distance").select(cols).show(5))
print("Longest by distance")
df_trips.orderBy(F.desc("trip_distance")).select(cols).show(5)

print("Shortest by time")
df_trips.orderBy("duration_min").select(cols).show(5)
print("Longest by time")
df_trips.orderBy(F.desc("duration_min")).select(cols).show(5)

Shortest by distance
+-----------+-------------+------------------+--------------------+---------------------+
|    trip_id|trip_distance|      duration_min|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------+-------------+------------------+--------------------+---------------------+
|60129542149|          0.0|2.6166666666666667| 2018-11-28 16:25:49|  2018-11-28 16:28:26|
|60129542150|          0.0|               4.1| 2018-11-28 16:29:37|  2018-11-28 16:33:43|
|60129542172|          0.0|               0.0| 2019-01-01 00:32:59|  2019-01-01 00:32:59|
|60129542148|          0.0|               1.6| 2018-11-28 15:56:57|  2018-11-28 15:58:33|
|60129542146|          0.0| 4.166666666666667| 2018-12-21 13:48:30|  2018-12-21 13:52:40|
+-----------+-------------+------------------+--------------------+---------------------+
only showing top 5 rows
Shortest real trip (distance > 0)
+-----------+-------------+-------------------+--------------------+---------------------+
|    trip_id|trip_di

In [59]:
#busiest day/slowest single day
df_jan = df_trips.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01")
    & (F.col("tpep_pickup_datetime") < "2019-02-01")
)
print("rows outside January 2019:", df_trips.count() - df_jan.count())

daily = df_jan.groupBy(
    F.to_date("tpep_pickup_datetime").alias("day")
).count()

print("Busiest days")
daily.orderBy(F.desc("count")).show(3)
print("Slowest days")
daily.orderBy("count").show(3)

rows outside January 2019: 537
Busiest days
+----------+------+
|       day| count|
+----------+------+
|2019-01-25|292499|
|2019-01-11|291714|
|2019-01-31|284625|
+----------+------+
only showing top 3 rows
Slowest days
+----------+------+
|       day| count|
+----------+------+
|2019-01-01|189432|
|2019-01-21|192826|
|2019-01-02|198737|
+----------+------+
only showing top 3 rows


In [60]:
#busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
df_jan = df_jan.withColumn("hour", F.hour("tpep_pickup_datetime"))

print("Trips per hour")
df_jan.groupBy("hour").count().orderBy("hour").show(24)

df_jan = df_jan.withColumn(
    "period",
    F.when((F.col("hour") >= 6) & (F.col("hour") < 12), "morning")
     .when((F.col("hour") >= 12) & (F.col("hour") < 17), "afternoon")
     .when((F.col("hour") >= 17) & (F.col("hour") < 22), "evening")
     .otherwise("late night")
)
print("Trips per period")
df_jan.groupBy("period").count().orderBy(F.desc("count")).show()

Trips per hour
+----+------+
|hour| count|
+----+------+
|   0|207758|
|   1|149242|
|   2|109413|
|   3| 78084|
|   4| 61423|
|   5| 75532|
|   6|178598|
|   7|304858|
|   8|373735|
|   9|365924|
|  10|361382|
|  11|375438|
|  12|401172|
|  13|404149|
|  14|433115|
|  15|452679|
|  16|420806|
|  17|468407|
|  18|515374|
|  19|475152|
|  20|423128|
|  21|409873|
|  22|369026|
|  23|281812|
+----+------+

Trips per period
+----------+-------+
|    period|  count|
+----------+-------+
|   evening|2291934|
| afternoon|2111921|
|   morning|1959935|
|late night|1332290|
+----------+-------+



In [61]:
#On average which day of the week is slowest/busiest
(daily
    .withColumn("weekday", F.date_format("day", "EEEE"))
    .groupBy("weekday")
    .agg(F.round(F.avg("count")).alias("avg_trips"),
         F.count("*").alias("nb_days"))
    .orderBy(F.desc("avg_trips"))
    .show())

+---------+---------+-------+
|  weekday|avg_trips|nb_days|
+---------+---------+-------+
|   Friday| 271788.0|      4|
| Thursday| 271398.0|      5|
|Wednesday| 253046.0|      5|
| Saturday| 252495.0|      4|
|  Tuesday| 241815.0|      5|
|   Monday| 226941.0|      4|
|   Sunday| 214973.0|      4|
+---------+---------+-------+



In [62]:
#Does trip distance or num passangers affect tip amount
card = df_jan.filter(F.col("payment_type") == 1)

print("corr distance / tip:",
      round(card.stat.corr("trip_distance", "tip_amount"), 3))
print("corr passengers / tip:",
      round(card.stat.corr("passenger_count", "tip_amount"), 3))

(card.groupBy("passenger_count")
    .agg(F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
         F.count("*").alias("nb_trips"))
    .orderBy("passenger_count")
    .show())

corr distance / tip: 0.672
corr passengers / tip: 0.01
+---------------+-------+--------+
|passenger_count|avg_tip|nb_trips|
+---------------+-------+--------+
|            0.0|   2.52|   83235|
|            1.0|   2.53| 3936707|
|            2.0|   2.61|  781267|
|            3.0|   2.59|  218514|
|            4.0|    2.6|   92063|
|            5.0|   2.62|  231253|
|            6.0|   2.61|  142893|
|            7.0|   11.3|      11|
|            8.0|   6.96|      27|
|            9.0|   3.51|       8|
+---------------+-------+--------+



In [63]:
#What was the highest "extra" charge and which trip
(df_trips.orderBy(F.desc("extra"))
    .select("trip_id", "extra", "fare_amount", "total_amount",
            "tpep_pickup_datetime")
    .show(5))

+-----------+------+-----------+------------+--------------------+
|    trip_id| extra|fare_amount|total_amount|tpep_pickup_datetime|
+-----------+------+-----------+------------+--------------------+
|60134865627|535.38|  355676.98|   356214.78| 2019-01-23 08:58:09|
|60136995374| 23.04|        4.5|       28.34| 2019-01-31 10:06:09|
|60129853196|  18.5|       52.0|        88.3| 2019-01-02 16:33:28|
|60131997230|  18.5|       49.0|       96.36| 2019-01-11 16:08:48|
|60129676693|  18.5|       39.5|        70.8| 2019-01-01 16:09:32|
+-----------+------+-----------+------------+--------------------+
only showing top 5 rows


In [64]:
#Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)
df_trips.describe(["passenger_count", "trip_distance", "fare_amount",
                   "extra", "tip_amount", "total_amount",
                   "duration_min"]).show()

df_trips.select(
    F.sum((F.col("passenger_count") == 0).cast("int")).alias("zero_pass"),
    F.sum((F.col("trip_distance") == 0).cast("int")).alias("zero_dist"),
    F.sum((F.col("total_amount") < 0).cast("int")).alias("neg_total"),
    F.sum((F.col("duration_min") <= 0).cast("int")).alias("dur_le_0"),
    F.sum((F.col("duration_min") > 24 * 60).cast("int")).alias("over_24h"),
).show()

+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+
|summary|   passenger_count|     trip_distance|      fare_amount|             extra|        tip_amount|     total_amount|      duration_min|
+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+
|  count|           7667945|           7696617|          7696617|           7696617|           7696617|          7696617|           7696617|
|   mean|1.5670317144945614|2.8301461681153532|12.52967677747685|0.3374054146126797|1.8208300763883147|15.81065134371489|16.551081570422276|
| stddev|1.2244198591042095| 3.774548394256295|261.5897471783846|0.5313564053935059|2.4994631914320986|261.8117056584905| 81.67539611217202|
|    min|               0.0|               0.0|           -362.0|             -60.0|             -63.5|           -362.8|          -84280.5|
|    max|    

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [65]:
#Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
# set dl url for the taxi zone lookup table
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

# get the data
response = requests.get(zone_url)

# check that response was good and save the data
zone_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(response.content)

# create the dataframe (csv, so we give the header and infer the schema)
df_zones = spark.read.csv(zone_file, header=True, inferSchema=True)
df_zones.printSchema()
df_zones.show(5)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [66]:
# Join the zone names to each trip (pickup and dropoff)
pu_zones = df_zones.select(F.col("LocationID").alias("PULocationID"),
                           F.col("Borough").alias("pu_borough"))
do_zones = df_zones.select(F.col("LocationID").alias("DOLocationID"),
                           F.col("Borough").alias("do_borough"))

df_b = (df_jan
    .join(pu_zones, on="PULocationID", how="left")
    .join(do_zones, on="DOLocationID", how="left"))

df_b.select("trip_id", "PULocationID", "pu_borough",
            "DOLocationID", "do_borough").show(5)

+-----------+------------+----------+------------+----------+
|    trip_id|PULocationID|pu_borough|DOLocationID|do_borough|
+-----------+------------+----------+------------+----------+
|60129542144|         151| Manhattan|         239| Manhattan|
|60129542145|         239| Manhattan|         246| Manhattan|
|60129542151|         163| Manhattan|         229| Manhattan|
|60129542152|         229| Manhattan|           7|    Queens|
|60129542153|         141| Manhattan|         234| Manhattan|
+-----------+------------+----------+------------+----------+
only showing top 5 rows


In [67]:
# which borough had most pickups? dropoffs?
print("Pickups by borough")
df_b.groupBy("pu_borough").count().orderBy(F.desc("count")).show()

print("Dropoffs by borough")
df_b.groupBy("do_borough").count().orderBy(F.desc("count")).show()

Pickups by borough
+-------------+-------+
|   pu_borough|  count|
+-------------+-------+
|    Manhattan|6950511|
|       Queens| 471113|
|      Unknown| 159807|
|     Brooklyn|  91896|
|        Bronx|  18056|
|          N/A|   3890|
|          EWR|    446|
|Staten Island|    361|
+-------------+-------+

Dropoffs by borough
+-------------+-------+
|   do_borough|  count|
+-------------+-------+
|    Manhattan|6816936|
|       Queens| 340914|
|     Brooklyn| 301074|
|      Unknown| 149091|
|        Bronx|  58068|
|          N/A|  16900|
|          EWR|  10913|
|Staten Island|   2184|
+-------------+-------+



In [68]:
# what are the busy/slow times by borough
from pyspark.sql import Window

# Number of trips per borough and per period of the day
df_b.groupBy("pu_borough").pivot("period").count().show()

# Busiest and slowest hour for each borough
hourly_b = df_b.groupBy("pu_borough", "hour").count()

w_busy = Window.partitionBy("pu_borough").orderBy(F.desc("count"))
w_slow = Window.partitionBy("pu_borough").orderBy("count")

(hourly_b
    .withColumn("rank_busy", F.row_number().over(w_busy))
    .withColumn("rank_slow", F.row_number().over(w_slow))
    .filter((F.col("rank_busy") == 1) | (F.col("rank_slow") == 1))
    .orderBy("pu_borough", F.desc("count"))
    .show())

+-------------+---------+-------+----------+-------+
|   pu_borough|afternoon|evening|late night|morning|
+-------------+---------+-------+----------+-------+
|       Queens|   130161| 140608|     92575| 107769|
|          EWR|      206|     95|        37|    108|
|      Unknown|    44971|  47472|     27498|  39866|
|     Brooklyn|    18870|  19131|     24469|  29426|
|Staten Island|       82|     70|        59|    150|
|          N/A|      870|    930|      1185|    905|
|    Manhattan|  1912221|2080710|   1183540|1774040|
|        Bronx|     4540|   2918|      2927|   7671|
+-------------+---------+-------+----------+-------+

+-------------+----+------+---------+---------+
|   pu_borough|hour| count|rank_busy|rank_slow|
+-------------+----+------+---------+---------+
|        Bronx|   7|  1803|        1|       24|
|        Bronx|   3|   225|       23|        1|
|     Brooklyn|   8|  6935|        1|       24|
|     Brooklyn|   3|  1919|       24|        1|
|          EWR|  15|    54|

In [69]:
# what are the busiest days of the week by borough?
daily_b = (df_b
    .groupBy("pu_borough", F.to_date("tpep_pickup_datetime").alias("day"))
    .count()
    .withColumn("weekday", F.date_format("day", "EEEE")))

avg_b = (daily_b.groupBy("pu_borough", "weekday")
    .agg(F.round(F.avg("count")).alias("avg_trips")))

avg_b.groupBy("pu_borough").pivot("weekday").sum("avg_trips").show()

w_day = Window.partitionBy("pu_borough").orderBy(F.desc("avg_trips"))
(avg_b.withColumn("rank", F.row_number().over(w_day))
    .filter(F.col("rank") == 1)
    .show())

+-------------+--------+--------+--------+--------+--------+--------+---------+
|   pu_borough|  Friday|  Monday|Saturday|  Sunday|Thursday| Tuesday|Wednesday|
+-------------+--------+--------+--------+--------+--------+--------+---------+
|       Queens| 16038.0| 16662.0| 11915.0| 14798.0| 15792.0| 15737.0|  15163.0|
|          EWR|    19.0|     8.0|    14.0|    17.0|    12.0|    15.0|     17.0|
|      Unknown|  5441.0|  5354.0|  5176.0|  4174.0|  5785.0|  4905.0|   5156.0|
|     Brooklyn|  3273.0|  2377.0|  2901.0|  2775.0|  3143.0|  3156.0|   3020.0|
|Staten Island|    16.0|    11.0|    10.0|    11.0|    12.0|    11.0|     10.0|
|          N/A|   111.0|   129.0|   123.0|   117.0|   127.0|   141.0|    127.0|
|    Manhattan|246224.0|201858.0|231875.0|192553.0|245903.0|217239.0| 228953.0|
|        Bronx|   667.0|   543.0|   482.0|   528.0|   624.0|   612.0|    600.0|
+-------------+--------+--------+--------+--------+--------+--------+---------+

+-------------+--------+---------+----+

In [70]:
# what is the average trip distance by borough?
(df_b.groupBy("pu_borough")
    .agg(F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
         F.count("*").alias("nb_trips"))
    .orderBy(F.desc("avg_distance"))
    .show())

+-------------+------------+--------+
|   pu_borough|avg_distance|nb_trips|
+-------------+------------+--------+
|Staten Island|        12.5|     361|
|       Queens|       11.28|  471113|
|        Bronx|        7.23|   18056|
|     Brooklyn|        4.79|   91896|
|          N/A|        3.19|    3890|
|          EWR|        2.64|     446|
|      Unknown|        2.42|  159807|
|    Manhattan|        2.23| 6950511|
+-------------+------------+--------+



In [71]:
# what is the average trip fare by borough?
(df_b.groupBy("pu_borough")
    .agg(F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.count("*").alias("nb_trips"))
    .orderBy(F.desc("avg_fare"))
    .show())

+-------------+--------+--------+
|   pu_borough|avg_fare|nb_trips|
+-------------+--------+--------+
|          EWR|   76.24|     446|
|          N/A|   59.57|    3890|
|Staten Island|   45.29|     361|
|       Queens|   35.14|  471113|
|        Bronx|   26.27|   18056|
|     Brooklyn|   18.65|   91896|
|      Unknown|   14.94|  159807|
|    Manhattan|   10.79| 6950511|
+-------------+--------+--------+



In [72]:
# highest/lowest fare amounts for a trip, what borough is associated with each
fare_cols = ["trip_id", "fare_amount", "trip_distance",
             "pu_borough", "do_borough"]

print("Highest fares")
df_b.orderBy(F.desc("fare_amount")).select(fare_cols).show(5)

print("Lowest fares")
df_b.orderBy("fare_amount").select(fare_cols).show(5)

Highest fares
+-----------+-----------+-------------+----------+----------+
|    trip_id|fare_amount|trip_distance|pu_borough|do_borough|
+-----------+-----------+-------------+----------+----------+
|60132041799|  623259.86|          2.4| Manhattan| Manhattan|
|60134865627|  355676.98|          0.0| Manhattan|   Unknown|
|60131702115|    36090.3|          0.0|   Unknown|   Unknown|
|60131434925|   34674.65|          0.0|   Unknown|   Unknown|
|60131191595|   33023.53|          0.0|   Unknown|   Unknown|
+-----------+-----------+-------------+----------+----------+
only showing top 5 rows
Lowest fares
+-----------+-----------+-------------+----------+----------+
|    trip_id|fare_amount|trip_distance|pu_borough|do_borough|
+-----------+-----------+-------------+----------+----------+
|60134432793|     -362.0|          0.0|    Queens|    Queens|
|60135850344|     -320.0|          0.0|       N/A|       N/A|
|60129599238|     -300.0|          0.0|     Bronx|     Bronx|
|60136769998|     -

In [73]:
# load the dataset from the most recently available january
recent_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet'

response = requests.get(recent_url)
print("status:", response.status_code)

recent_file = "yellow_tripdata_2026-01.parquet"
if response.status_code == 200:
    with open(recent_file, "wb") as f:
        f.write(response.content)

df_recent = spark.read.parquet(recent_file)


def month_metrics(df, start, end, label):
    """Average metrics for trips picked up between start and end."""
    return (df
        .filter((F.col("tpep_pickup_datetime") >= start)
                & (F.col("tpep_pickup_datetime") < end))
        .withColumn("duration_min",
                    (F.unix_timestamp("tpep_dropoff_datetime")
                     - F.unix_timestamp("tpep_pickup_datetime")) / 60)
        .agg(F.count("*").alias("nb_trips"),
             F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
             F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
             F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
             F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
             F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
             F.round(F.avg("total_amount"), 2).alias("avg_total"))
        .withColumn("dataset", F.lit(label)))


(month_metrics(df_trips, "2019-01-01", "2019-02-01", "jan 2019")
    .unionByName(month_metrics(df_recent, "2026-01-01", "2026-02-01",
                               "jan 2026"))
    .show())

status: 200
+--------+--------------+------------+----------------+--------+-------+---------+--------+
|nb_trips|avg_passengers|avg_distance|avg_duration_min|avg_fare|avg_tip|avg_total| dataset|
+--------+--------------+------------+----------------+--------+-------+---------+--------+
| 7696080|          1.57|        2.83|           16.54|   12.53|   1.82|    15.81|jan 2019|
| 3724882|          1.26|        6.46|           17.19|    20.8|   2.61|    29.18|jan 2026|
+--------+--------------+------------+----------------+--------+-------+---------+--------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [74]:
# Register the DataFrames as temporary SQL tables
# so they can be used by name in SQL queries
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [75]:
# Average passenger count (same as the PySpark version)
spark.sql("""
    SELECT ROUND(AVG(passenger_count), 2) AS avg_passengers  -- mean, 2 decimals
    FROM trips                                               -- all trips
""").show()

+--------------+
|avg_passengers|
+--------------+
|          1.57|
+--------------+



In [76]:
# Number of trips per day, busiest first
spark.sql("""
    SELECT TO_DATE(tpep_pickup_datetime) AS day, COUNT(*) AS nb_trips
    FROM trips
    WHERE tpep_pickup_datetime BETWEEN '2019-01-01' AND '2019-02-01'
    GROUP BY day
    ORDER BY nb_trips DESC
""").show(31)

+----------+--------+
|       day|nb_trips|
+----------+--------+
|2019-01-25|  292499|
|2019-01-11|  291714|
|2019-01-31|  284625|
|2019-01-17|  284580|
|2019-01-24|  281959|
|2019-01-10|  281863|
|2019-01-30|  276774|
|2019-01-16|  272699|
|2019-01-26|  271993|
|2019-01-15|  267370|
|2019-01-18|  266848|
|2019-01-12|  265115|
|2019-01-23|  261151|
|2019-01-29|  259786|
|2019-01-09|  255868|
|2019-01-22|  255178|
|2019-01-14|  245082|
|2019-01-28|  241040|
|2019-01-08|  237310|
|2019-01-05|  236506|
|2019-01-19|  236365|
|2019-01-04|  236089|
|2019-01-07|  228816|
|2019-01-13|  227502|
|2019-01-03|  223965|
|2019-01-27|  220451|
|2019-01-06|  208823|
|2019-01-20|  203114|
|2019-01-02|  198737|
|2019-01-21|  192826|
|2019-01-01|  189432|
+----------+--------+
only showing top 31 rows


In [77]:
# Number of pickups per borough, using a join with the zones table
spark.sql("""
    SELECT zones.Borough, COUNT(*) AS nb_pickups
    FROM trips
    JOIN zones ON trips.PULocationID = zones.LocationID
    GROUP BY zones.Borough
    ORDER BY nb_pickups DESC
""").show()

+-------------+----------+
|      Borough|nb_pickups|
+-------------+----------+
|    Manhattan|   6950965|
|       Queens|    471173|
|      Unknown|    159815|
|     Brooklyn|     91905|
|        Bronx|     18062|
|          N/A|      3890|
|          EWR|       446|
|Staten Island|       361|
+-------------+----------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing